# 20 — Convert Full Locked PyTorch Ensemble → One Keras Core Model

Mục tiêu: **không train lại**. Convert toàn bộ locked ensemble hiện tại sang **một file `.keras` duy nhất** và chứng minh output Keras gần như trùng ensemble PyTorch.

Locked ensemble:
- DenseNet121, 224×224
- force grayscale → RGB
- 5 folds seed 2026
- 5 folds seed 3407
- group weights đọc trực tiếp từ `locked_pipeline.json`
- threshold đọc trực tiếp từ `locked_pipeline.json`
- TTA OFF

Model tạo ở notebook này là **CORE model**:
- Input: tensor float32 đã preprocess/normalize, shape `(N,224,224,3)`
- Output: 2-class probabilities `[P(Normal), P(Pneumonia)]`

Chưa đóng gói raw-image preprocessing vào model vì interface BTC vẫn cần xác nhận. Sau khi core parity PASS, ta mới tạo wrapper final phù hợp cách BTC gọi `model.predict()`.

### Numerical parity rule
PyTorch reference inference is forced to **CPU**, matching TensorFlow/Keras on native Windows. Notebook 19 already showed CPU↔CPU parity at ~4e-7. GPU↔CPU drift is a backend numerical effect and is not a weight-conversion error.

### Gate
1. Mỗi checkpoint PyTorch ↔ Keras phải có max probability diff ≤ 1e-4.
2. Full ensemble trước save phải match PyTorch.
3. Save → `tf.keras.models.load_model(...)` → output vẫn match.
4. Nếu bất kỳ gate nào fail: dừng, không dùng file đó để nộp.


In [ ]:
print('>>> CELL 1 START: environment + locked config', flush=True)
import os,sys,json,gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torchvision import models,transforms
import tensorflow as tf

ROOT=Path(r'D:\medicine')
LOCK_PATH=ROOT/'cv_outputs'/'final_ensemble_lock'/'locked_pipeline.json'
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'conversion_outputs'/'full_keras_ensemble'
OUT.mkdir(parents=True,exist_ok=True)

KERAS_PATH=OUT/'locked_densenet10_ensemble_core.keras'
SUMMARY_PATH=OUT/'full_ensemble_conversion_summary.json'

N_SAMPLES=16
PT_BATCH=4
KERAS_BATCH=1
IMAGE_SIZE=224
TOL=1e-4
PT_DEVICE='cpu'  # IMPORTANT: match TensorFlow/Keras CPU for conversion parity

assert LOCK_PATH.exists(),f'Missing lock: {LOCK_PATH}'
assert MANIFEST.exists(),f'Missing manifest: {MANIFEST}'

with open(LOCK_PATH,'r',encoding='utf-8') as f:
    lock=json.load(f)

assert lock['architecture'].lower()=='densenet121'
assert int(lock['image_size'])==224
assert lock['preprocessing']=='force_grayscale_then_rgb'
assert lock['tta'] is False
assert len(lock['models'])==2

THRESHOLD=float(lock['threshold'])
GROUPS=lock['models']

print('Python:',sys.version.split()[0],flush=True)
print('PyTorch:',torch.__version__,flush=True)
print('TensorFlow:',tf.__version__,flush=True)
print('PyTorch parity device:',PT_DEVICE,flush=True)
print('CUDA available (not used for parity):',torch.cuda.is_available(),flush=True)
print('Threshold:',THRESHOLD,flush=True)
print('Groups:',[(g['name'],g['weight']) for g in GROUPS],flush=True)
print('Group weight sum:',sum(float(g['weight']) for g in GROUPS),flush=True)

assert abs(sum(float(g['weight']) for g in GROUPS)-1.0)<1e-8

# Resolve 10 checkpoint paths from the frozen lock.
CHECKPOINT_GROUPS=[]
for g in GROUPS:
    d=Path(g['checkpoint_dir'])
    if not d.exists():
        d=ROOT/'cv_outputs'/d.name
    ckpts=[d/f'fold{i}_best.pt' for i in range(5)]
    missing=[str(p) for p in ckpts if not p.exists()]
    assert not missing,f'Missing checkpoints: {missing}'
    CHECKPOINT_GROUPS.append({
        'name':g['name'],
        'weight':float(g['weight']),
        'dir':d,
        'checkpoints':ckpts,
    })

print('\nCheckpoint preflight:',flush=True)
for g in CHECKPOINT_GROUPS:
    print(f"  {g['name']} weight={g['weight']:.6f}",flush=True)
    for p in g['checkpoints']:
        print('   ',p,flush=True)


In [ ]:
print('>>> CELL 2 START: identical normalized sample tensors', flush=True)

manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy()
sample_df=dev[dev.cv_fold==0].head(N_SAMPLES).copy().reset_index(drop=True)
assert len(sample_df)==N_SAMPLES

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        s=max(w,h)
        l=(s-w)//2;r=s-w-l
        t=(s-h)//2;b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pt_tf=transforms.Compose([
    PadToSquare(),
    transforms.Resize(
        (IMAGE_SIZE,IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

xs=[]
names=[]
for _,r in sample_df.iterrows():
    with Image.open(r.path) as im:
        x=pt_tf(im.convert('L').convert('RGB'))
    xs.append(x)
    names.append(r.filename)

x_pt=torch.stack(xs,dim=0)
x_keras=np.transpose(x_pt.numpy(),(0,2,3,1)).astype(np.float32)

print('PyTorch input:',tuple(x_pt.shape),flush=True)
print('Keras input:',x_keras.shape,flush=True)
print('Sample filenames:',names[:5], '...',flush=True)


In [ ]:
print('>>> CELL 3 START: exact torchvision-style Keras DenseNet builder', flush=True)

L=tf.keras.layers

def bn(x,name,layer_map):
    layer=L.BatchNormalization(
        axis=-1,
        momentum=0.1,
        epsilon=1e-5,
        center=True,
        scale=True,
        name=name.replace('.','_')
    )
    y=layer(x)
    layer_map[name]=layer
    return y

def conv(x,filters,kernel,strides=1,name=None,layer_map=None):
    layer=L.Conv2D(
        filters,
        kernel_size=kernel,
        strides=strides,
        padding='valid',
        use_bias=False,
        data_format='channels_last',
        name=name.replace('.','_')
    )
    y=layer(x)
    layer_map[name]=layer
    return y

def build_torchvision_densenet121_keras(
    input_shape=(224,224,3),
    classes=2,
    model_name='densenet121'
):
    layer_map={}
    inp=tf.keras.Input(shape=input_shape,dtype=tf.float32,name='image')

    x=L.ZeroPadding2D(padding=3,name='features_conv0_pad')(inp)
    x=conv(x,64,7,strides=2,name='features.conv0',layer_map=layer_map)
    x=bn(x,'features.norm0',layer_map)
    x=L.ReLU(name='features_relu0')(x)

    x=L.ZeroPadding2D(padding=1,name='features_pool0_pad')(x)
    x=L.MaxPooling2D(pool_size=3,strides=2,padding='valid',name='features_pool0')(x)

    block_config=(6,12,24,16)
    growth_rate=32
    bn_size=4
    num_features=64

    for block_idx,num_layers in enumerate(block_config,start=1):
        features=[x]

        for layer_idx in range(1,num_layers+1):
            cat=features[0] if len(features)==1 else L.Concatenate(
                axis=-1,
                name=f'features_denseblock{block_idx}_denselayer{layer_idx}_input_concat'
            )(features)

            prefix=f'features.denseblock{block_idx}.denselayer{layer_idx}'

            y=bn(cat,prefix+'.norm1',layer_map)
            y=L.ReLU(name=(prefix+'.relu1').replace('.','_'))(y)
            y=conv(y,bn_size*growth_rate,1,name=prefix+'.conv1',layer_map=layer_map)

            y=bn(y,prefix+'.norm2',layer_map)
            y=L.ReLU(name=(prefix+'.relu2').replace('.','_'))(y)
            y=L.ZeroPadding2D(
                padding=1,
                name=(prefix+'.conv2_pad').replace('.','_')
            )(y)
            y=conv(y,growth_rate,3,name=prefix+'.conv2',layer_map=layer_map)

            features.append(y)

        x=L.Concatenate(
            axis=-1,
            name=f'features_denseblock{block_idx}_output_concat'
        )(features)
        num_features += num_layers*growth_rate

        if block_idx != 4:
            prefix=f'features.transition{block_idx}'
            x=bn(x,prefix+'.norm',layer_map)
            x=L.ReLU(name=(prefix+'.relu').replace('.','_'))(x)
            x=conv(x,num_features//2,1,name=prefix+'.conv',layer_map=layer_map)
            x=L.AveragePooling2D(
                pool_size=2,
                strides=2,
                padding='valid',
                name=(prefix+'.pool').replace('.','_')
            )(x)
            num_features//=2

    x=bn(x,'features.norm5',layer_map)
    x=L.ReLU(name='features_relu5')(x)
    x=L.GlobalAveragePooling2D(name='adaptive_avg_pool')(x)

    classifier=L.Dense(classes,use_bias=True,name='classifier')
    logits=classifier(x)
    layer_map['classifier']=classifier

    model=tf.keras.Model(inp,logits,name=model_name)
    return model,layer_map

def map_state_to_keras(state,k_model,layer_map):
    def arr(key):
        return state[key].detach().cpu().numpy()

    mapped=[]
    for prefix,layer in layer_map.items():
        if prefix=='classifier':
            layer.set_weights([
                arr('classifier.weight').T,
                arr('classifier.bias')
            ])
            mapped += ['classifier.weight','classifier.bias']

        elif isinstance(layer,tf.keras.layers.Conv2D):
            w=np.transpose(arr(prefix+'.weight'),(2,3,1,0))
            layer.set_weights([w])
            mapped.append(prefix+'.weight')

        elif isinstance(layer,tf.keras.layers.BatchNormalization):
            layer.set_weights([
                arr(prefix+'.weight'),
                arr(prefix+'.bias'),
                arr(prefix+'.running_mean'),
                arr(prefix+'.running_var')
            ])
            mapped += [
                prefix+'.weight',
                prefix+'.bias',
                prefix+'.running_mean',
                prefix+'.running_var'
            ]
        else:
            raise TypeError((prefix,type(layer)))

    expected=[
        k for k in state.keys()
        if not k.endswith('num_batches_tracked')
    ]
    assert set(mapped)==set(expected),(
        'State mapping mismatch',
        sorted(set(expected)-set(mapped))[:10],
        sorted(set(mapped)-set(expected))[:10]
    )

print('Builder ready.',flush=True)


In [ ]:
print('>>> CELL 4 START: sequentially convert + verify all 10 checkpoints', flush=True)

individual_reports=[]
keras_groups=[]
pt_group_probabilities=[]

for group_idx,g in enumerate(CHECKPOINT_GROUPS):
    print(f"\n=== GROUP {g['name']} weight={g['weight']:.6f} ===",flush=True)

    group_models=[]
    group_pt_probs=[]

    for fold,ckpt_path in enumerate(g['checkpoints']):
        tag=f"{g['name']}_fold{fold}"
        print(f"\n[{tag}] load PyTorch",flush=True)

        ck=torch.load(ckpt_path,map_location='cpu')
        state=ck['state_dict']

        # Reference PyTorch model, one checkpoint at a time.
        # CPU is intentional: TensorFlow/Keras is CPU on native Windows.
        # Comparing CUDA PyTorch against CPU Keras introduces backend numerical drift.
        pt_model=models.densenet121(weights=None)
        pt_model.classifier=nn.Linear(pt_model.classifier.in_features,2)
        pt_model.load_state_dict(state,strict=True)
        pt_model.eval().to(PT_DEVICE)

        pt_parts=[]
        with torch.inference_mode():
            for start in range(0,N_SAMPLES,PT_BATCH):
                end=min(start+PT_BATCH,N_SAMPLES)
                xb=x_pt[start:end].to(PT_DEVICE)
                logits=pt_model(xb)
                prob=torch.softmax(logits,dim=1).detach().cpu().numpy()
                pt_parts.append(prob)
                del xb,logits,prob
        pt_prob=np.concatenate(pt_parts,axis=0)
        group_pt_probs.append(pt_prob)

        # Convert this checkpoint to a nested Keras submodel.
        k_model,layer_map=build_torchvision_densenet121_keras(
            model_name=tag
        )
        map_state_to_keras(state,k_model,layer_map)

        # Per-checkpoint parity on all sample images, one image at a time.
        k_parts=[]
        for start in range(0,N_SAMPLES,KERAS_BATCH):
            end=min(start+KERAS_BATCH,N_SAMPLES)
            out=k_model(x_keras[start:end],training=False).numpy()
            k_parts.append(tf.nn.softmax(out,axis=1).numpy())
            del out
        k_prob=np.concatenate(k_parts,axis=0)

        diff=np.abs(pt_prob-k_prob)
        rep={
            'group':g['name'],
            'fold':int(fold),
            'checkpoint':str(ckpt_path),
            'max_abs_probability_diff':float(diff.max()),
            'mean_abs_probability_diff':float(diff.mean()),
            'class_agreement_at_0_5':float(
                (pt_prob.argmax(1)==k_prob.argmax(1)).mean()
            ),
            'pass':bool(
                diff.max()<=TOL and
                (pt_prob.argmax(1)==k_prob.argmax(1)).all()
            )
        }
        individual_reports.append(rep)

        print(json.dumps(rep,indent=2),flush=True)
        assert rep['pass'],f'Checkpoint parity FAIL: {tag}'

        group_models.append(k_model)

        # Release PyTorch copy; Keras submodel is intentionally retained.
        del pt_model,ck,state,layer_map,k_prob,diff,pt_parts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    group_pt=np.mean(np.stack(group_pt_probs,axis=0),axis=0)
    pt_group_probabilities.append(group_pt)
    keras_groups.append({
        'name':g['name'],
        'weight':g['weight'],
        'models':group_models,
    })

    del group_pt_probs
    gc.collect()

print('\nALL 10 CHECKPOINT PARITY TESTS: PASS',flush=True)


In [ ]:
print('>>> CELL 5 START: build one standard-layer Keras ensemble', flush=True)

inp=tf.keras.Input(
    shape=(IMAGE_SIZE,IMAGE_SIZE,3),
    dtype=tf.float32,
    name='normalized_image'
)

weighted_group_outputs=[]

for g in keras_groups:
    fold_probs=[]

    for fold,k_model in enumerate(g['models']):
        logits=k_model(inp,training=False)
        probs=L.Softmax(
            axis=-1,
            name=f"{g['name']}_fold{fold}_softmax"
        )(logits)
        fold_probs.append(probs)

    group_avg=L.Average(
        name=f"{g['name']}_five_fold_average"
    )(fold_probs)

    weighted=L.Rescaling(
        scale=float(g['weight']),
        offset=0.0,
        name=f"{g['name']}_weight_{str(g['weight']).replace('.','p')}"
    )(group_avg)

    weighted_group_outputs.append(weighted)

ensemble_probs=L.Add(
    name='ensemble_probabilities'
)(weighted_group_outputs)

ensemble_model=tf.keras.Model(
    inp,
    ensemble_probs,
    name='locked_densenet121_10model_ensemble_core'
)

print('Total params:',ensemble_model.count_params(),flush=True)
print('Input:',ensemble_model.input_shape,flush=True)
print('Output:',ensemble_model.output_shape,flush=True)


In [ ]:
print('>>> CELL 6 START: full PyTorch ensemble vs Keras ensemble BEFORE save', flush=True)

# PyTorch reference: five-fold mean inside each seed, then locked seed weights.
pt_ensemble=np.zeros_like(pt_group_probabilities[0],dtype=np.float64)
for g,p in zip(CHECKPOINT_GROUPS,pt_group_probabilities):
    pt_ensemble += float(g['weight'])*p

# Keras full ensemble, micro-batched to keep RAM stable.
k_parts=[]
for start in range(0,N_SAMPLES,KERAS_BATCH):
    end=min(start+KERAS_BATCH,N_SAMPLES)
    out=ensemble_model(x_keras[start:end],training=False).numpy()
    k_parts.append(out)
    del out
    gc.collect()
    print(f'  ensemble processed {end}/{N_SAMPLES}',flush=True)

k_ensemble=np.concatenate(k_parts,axis=0)
full_diff=np.abs(pt_ensemble-k_ensemble)

pt_p=pt_ensemble[:,1]
k_p=k_ensemble[:,1]
pt_pred=(pt_p>=THRESHOLD).astype(int)
k_pred=(k_p>=THRESHOLD).astype(int)

before_save={
    'n_samples':int(N_SAMPLES),
    'max_abs_probability_diff_all_classes':float(full_diff.max()),
    'mean_abs_probability_diff_all_classes':float(full_diff.mean()),
    'max_abs_pneumonia_probability_diff':float(np.abs(pt_p-k_p).max()),
    'class_agreement_at_locked_threshold':float((pt_pred==k_pred).mean()),
    'threshold':float(THRESHOLD),
}

PASS_BEFORE=(
    before_save['max_abs_probability_diff_all_classes']<=TOL
    and before_save['class_agreement_at_locked_threshold']==1.0
)

print(json.dumps(before_save,indent=2),flush=True)
print('FULL ENSEMBLE PARITY BEFORE SAVE:', 'PASS' if PASS_BEFORE else 'FAIL',flush=True)
assert PASS_BEFORE,'Full ensemble parity failed before save.'

cmp=pd.DataFrame({
    'filename':names,
    'pt_p_pneumonia':pt_p,
    'keras_p_pneumonia':k_p,
    'abs_diff':np.abs(pt_p-k_p),
    'pt_pred_locked_t':pt_pred,
    'keras_pred_locked_t':k_pred,
})
cmp.to_csv(OUT/'full_ensemble_parity_before_save.csv',index=False)
display(cmp.sort_values('abs_diff',ascending=False))


In [ ]:
print('>>> CELL 7 START: save one .keras file', flush=True)

ensemble_model.save(KERAS_PATH)
size_mb=KERAS_PATH.stat().st_size/(1024**2)

print('Saved:',KERAS_PATH,flush=True)
print(f'File size: {size_mb:.2f} MiB',flush=True)


In [ ]:
print('>>> CELL 8 START: free original graphs + load_model()', flush=True)

# Preserve only small NumPy reference arrays/reports.
del ensemble_model
del keras_groups
del weighted_group_outputs
del k_parts
gc.collect()
tf.keras.backend.clear_session()
gc.collect()

print('Loading with plain tf.keras.models.load_model(...), no custom_objects',flush=True)
reloaded=tf.keras.models.load_model(KERAS_PATH,compile=False)
print('Reloaded:',reloaded.name,flush=True)
print('Input:',reloaded.input_shape,flush=True)
print('Output:',reloaded.output_shape,flush=True)


In [ ]:
print('>>> CELL 9 START: parity AFTER reload', flush=True)

reload_parts=[]
for start in range(0,N_SAMPLES,KERAS_BATCH):
    end=min(start+KERAS_BATCH,N_SAMPLES)
    out=reloaded(x_keras[start:end],training=False).numpy()
    reload_parts.append(out)
    del out
    gc.collect()
    print(f'  reload processed {end}/{N_SAMPLES}',flush=True)

reload_probs=np.concatenate(reload_parts,axis=0)
reload_diff=np.abs(pt_ensemble-reload_probs)

reload_p=reload_probs[:,1]
reload_pred=(reload_p>=THRESHOLD).astype(int)

after_reload={
    'max_abs_probability_diff_all_classes_vs_pytorch':float(reload_diff.max()),
    'mean_abs_probability_diff_all_classes_vs_pytorch':float(reload_diff.mean()),
    'max_abs_pneumonia_probability_diff_vs_pytorch':float(
        np.abs(pt_p-reload_p).max()
    ),
    'class_agreement_at_locked_threshold_vs_pytorch':float(
        (pt_pred==reload_pred).mean()
    ),
    'max_abs_diff_vs_keras_before_save':float(
        np.abs(k_ensemble-reload_probs).max()
    ),
}

PASS_RELOAD=(
    after_reload['max_abs_probability_diff_all_classes_vs_pytorch']<=TOL
    and after_reload['class_agreement_at_locked_threshold_vs_pytorch']==1.0
    and after_reload['max_abs_diff_vs_keras_before_save']<=1e-7
)

print(json.dumps(after_reload,indent=2),flush=True)
print('FULL ENSEMBLE KERAS RELOAD PARITY:', 'PASS' if PASS_RELOAD else 'FAIL',flush=True)
assert PASS_RELOAD,'Reloaded full ensemble parity failed.'


In [ ]:
print('>>> CELL 10 START: final conversion summary', flush=True)

summary={
    'status':'PASS' if (PASS_BEFORE and PASS_RELOAD) else 'FAIL',
    'source_lock':str(LOCK_PATH),
    'keras_core_file':str(KERAS_PATH),
    'keras_file_size_mib':float(size_mb),
    'input_contract':{
        'dtype':'float32',
        'shape':[None,224,224,3],
        'meaning':'already preprocessed: pad-square, resize 224, grayscale replicated to RGB, ImageNet normalized'
    },
    'output_contract':{
        'shape':[None,2],
        'order':['Normal','Pneumonia'],
        'type':'probabilities',
        'locked_threshold_for_pneumonia':float(THRESHOLD)
    },
    'ensemble':[
        {
            'name':g['name'],
            'weight':float(g['weight']),
            'n_folds':5
        }
        for g in CHECKPOINT_GROUPS
    ],
    'individual_checkpoint_parity':individual_reports,
    'before_save':before_save,
    'after_reload':after_reload,
    'safe_to_build_final_btc_wrapper':bool(PASS_BEFORE and PASS_RELOAD)
}

with open(SUMMARY_PATH,'w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2),flush=True)

print('\nFINAL RESULT:',flush=True)
if summary['safe_to_build_final_btc_wrapper']:
    print('PASS — full locked 10-checkpoint ensemble reproduced as ONE loadable .keras core model.',flush=True)
    print('NEXT — build BTC-facing preprocessing/output wrapper only after input interface is confirmed.',flush=True)
else:
    print('FAIL — do not use this Keras model.',flush=True)


## Sau khi chạy

Chạy:

```powershell
Get-Content .\conversion_outputs\full_keras_ensemble\full_ensemble_conversion_summary.json
```

Gửi lại phần:
- `status`
- `keras_file_size_mib`
- `before_save`
- `after_reload`
- `safe_to_build_final_btc_wrapper`

Nếu PASS, **không train lại**. Bước tiếp theo chỉ còn làm wrapper đúng interface BTC + file nộp `.pkl/.json/.ipynb/team01.csv`.
